# CineMovie — Exploratory Data Analysis

This notebook inspects the TMDB 5000 Movie Dataset before recommendation-model development.

In [ ]:
from pathlib import Path
from src.data_loader import load_dataset

MOVIES_PATH = Path('../data/tmdb_5000_movies.csv')
CREDITS_PATH = Path('../data/tmdb_5000_credits.csv')

df = load_dataset(MOVIES_PATH, CREDITS_PATH)
df.shape

In [ ]:
# Dataset structure
df.info()
df.head()

In [ ]:
# Missing values and duplicates
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0])
print('Duplicate rows:', df.duplicated().sum())

In [ ]:
# Rating and popularity distributions
display(df[['vote_average', 'vote_count', 'popularity']].describe())

In [ ]:
# JSON-like metadata fields
from src.preprocessing import parse_json_names

for column in ['genres', 'keywords']:
    parsed = df[column].apply(parse_json_names)
    print(f'{column}: empty rows = {(parsed.str.len() == 0).sum()}')

## Initial data-quality decisions

- Keep `overview`, `tagline`, `genres`, `keywords`, `cast`, and `director` usable even when individual fields are missing by filling them with empty values rather than dropping the whole movie.
- Keep `vote_average` and `vote_count` for later quality-aware re-ranking.
- Do not use `homepage`, budget, revenue, or production-company fields as core recommendation features in the first model.
- Parse genres/keywords/cast/crew JSON strings into model-ready text before TF-IDF.
- Inspect and handle zero-vote movies carefully before using rating-based ranking.